In [8]:
%%writefile tsp_domain.py
import sys
import os

# ==========================================
# 1. CRITICAL PATH SETUP
# ==========================================
PROJECT_ROOT = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP"
LIB_PATH = os.path.join(PROJECT_ROOT, "Evolutionary_algorithm")

if os.path.exists(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

if os.path.exists(LIB_PATH) and LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)

# ==========================================
# 2. IMPORTS
# ==========================================
import numpy as np
import modified_didppy as m_dp
from ortools.linear_solver import pywraplp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
from functools import lru_cache

try:
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
except ImportError:
    pass 

# ==========================================
# 3. GLOBALS & READER
# ==========================================
current_num_locations = 0
current_travel_cost = []

def read_tsp_cappart_format(file_path):
    with open(file_path, 'r') as f:
        values = f.read().split()
    iterator = iter(values)
    try:
        n = int(next(iterator))
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                row.append(int(float(next(iterator))))
            c.append(row)
        return n, c
    except StopIteration:
        return 0, []

Overwriting tsp_domain.py


In [9]:
%%writefile -a tsp_domain.py

# ==========================================
# 4. MODEL DEFINITION
# ==========================================
def creation_of_didp_model_function():
    n = current_num_locations
    c = current_travel_cost
    
    model = m_dp.Model(maximize=False, float_cost=True)
    customer = model.add_object_type(number=n)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    location = model.add_element_var(object_type=customer, target=0)
    travel_time = model.add_float_table(c)

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=travel_time[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[(unvisited, unvisited.remove(j)), (location, j)],
        )
        model.add_transition(visit)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location, 0)],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)
    model.add_base_case([unvisited.is_empty(), location == 0])
    
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
    }
    return model, metadata

Appending to tsp_domain.py


In [10]:
%%writefile -a tsp_domain.py

# ==========================================
# 5. DUAL BOUNDS
# ==========================================

def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0
    infinity = solver.infinity()

    x = {}
    u = {}
    for i in range(n_nodes):
        u[i] = solver.NumVar(0, n_nodes, f'u_{i}')
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
                c.SetCoefficient(u[i], 1)
                c.SetCoefficient(u[j], -1)
                c.SetCoefficient(x[(i, j)], n_nodes)

    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    @lru_cache(maxsize=10000)
    def _solve_3idx(active_tuple):
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0) 
        max_u = len(active_set) 

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(0, 0); u[i].SetBounds(0, 0)
                elif i == 0:
                    cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(1, 1); u[i].SetBounds(1, max_u)
                else:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(1, 1); u[i].SetBounds(1, max_u)
            else:
                cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(0, 0); u[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_3idx(key)

    return h_lp_relaxation_3_idx

def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0

    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    @lru_cache(maxsize=10000)
    def _solve_2idx(active_tuple):
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0)

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(0, 0)
                elif i == 0:
                    cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(1, 1)
                else:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(1, 1)
            else:
                cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_2idx(key)

    return h_lp_relaxation_2_idx

def dual_bound_expression_function(didp_bundle):
    model, metadata = didp_bundle
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    cost_matrix = np.array(metadata['distance_matrix'])
    
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    min_outgoing_arr = np.min(masked_cost, axis=1)
    min_incoming_arr = np.min(masked_cost, axis=0)

    h_lp_relaxation_3_idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_relaxation_2_idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)

    @lru_cache(maxsize=100000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        sum_in = np.sum(mins_in[1:])
        sum_out = np.sum(mins_out[:-1])
        return float(0.5 * (sum_in + sum_out))

    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        active_list = [curr] + sorted(list(U))
        if 0 not in active_list: active_list.append(0)
        return _calc_degree(tuple(active_list))

    @lru_cache(maxsize=100000)
    def _calc_min_flow_static(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            val_in += min_incoming_arr[0]
        return float(max(val_out, val_in))

    @lru_cache(maxsize=100000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_mst(state):
        U = state[unvisited_var]
        return _calc_mst(tuple(sorted(list(U))))

    @lru_cache(maxsize=100000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        if len(subset) > 1:
            sub_mat = cost_matrix[np.ix_(subset, subset)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0
        return float(mst_val + e1 + e2)

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    @lru_cache(maxsize=100000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row, col = linear_sum_assignment(assign_mat)
        return float(assign_mat[row, col].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    @lru_cache(maxsize=100000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        N = len(nodes)
        if N < 2: return 0.0
        
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals): phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    return automatic_creation_of_dual_bounds_registry(locals())

Appending to tsp_domain.py


In [13]:
%%writefile worker.py
import sys
import os
import ast
import json
import time
import threading

# --- CONFIGURATION ---
MEMORY_LIMIT_MB = 7990  # Limit worker to 4 GB RAM. Adjust based on your PC.

# --- 1. SETUP PATHS ---
PROJECT_ROOT = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP"
LIB_PATH = os.path.join(PROJECT_ROOT, "Evolutionary_algorithm")

if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
if LIB_PATH not in sys.path: sys.path.append(LIB_PATH)

# --- 2. IMPORTS ---
try:
    import psutil # For memory monitoring
    import modified_didppy as m_dp
    from evolutionary_algorithm_lib import combining_modified_didppy_solver_with_chromosome
    import tsp_domain 
except ImportError as e:
    print(json.dumps({"status": "error", "message": f"Import Error: {e}"}))
    sys.exit(1)

# --- 3. MEMORY GUARD (Background Thread) ---
def memory_guard():
    """Checks memory usage every second and kills process if limit exceeded."""
    process = psutil.Process(os.getpid())
    limit_bytes = MEMORY_LIMIT_MB * 1024 * 1024
    
    while True:
        try:
            mem_usage = process.memory_info().rss # Resident Set Size (Physical Memory)
            if mem_usage > limit_bytes:
                # Print a special error message that the logs will catch
                sys.stderr.write(f"\n[Guard] Memory limit ({MEMORY_LIMIT_MB}MB) reached. Killing worker.\n")
                sys.stderr.flush()
                os._exit(1) # Force kill immediately (simulates a crash)
            time.sleep(1)
        except:
            break

# --- 4. MAIN LOGIC ---
if __name__ == "__main__":
    try:
        # Start Memory Guard
        guard_thread = threading.Thread(target=memory_guard, daemon=True)
        guard_thread.start()

        if len(sys.argv) < 5:
            raise ValueError("Not enough arguments")
            
        instance_name = sys.argv[1]
        chromosome_str = sys.argv[2]
        data_dir = sys.argv[3]
        time_limit = float(sys.argv[4])

        file_path = os.path.join(data_dir, instance_name)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        n_val, c_val = tsp_domain.read_tsp_cappart_format(file_path)
        tsp_domain.current_num_locations = n_val
        tsp_domain.current_travel_cost = c_val
        
        start_time = time.time()
        result = combining_modified_didppy_solver_with_chromosome(
            chromosome=ast.literal_eval(chromosome_str),
            didp_model_registry=tsp_domain.creation_of_didp_model_function,
            dual_bound_expression_function=tsp_domain.dual_bound_expression_function,
            solver_time_limit=time_limit,
            output_other_result=True,
            print_timing_stats=True,
            solver_quite=False 
        )
        duration = time.time() - start_time

        if result is None:
            output = {"status": "timeout", "cost": float('inf'), "duration": duration}
        elif isinstance(result, (float, int)): 
             output = {"status": "success", "cost": float(result), "duration": duration}
        else: 
            cost, is_opt, gen, exp, stats = result
            output = {
                "status": "success",
                "cost": cost,
                "is_optimal": is_opt,
                "generated": gen,
                "expanded": exp,
                "duration": duration,
                "stats": stats
            }
        
        print(json.dumps(output))

    except Exception as e:
        print(json.dumps({"status": "error", "message": str(e)}))
        sys.exit(1)

Overwriting worker.py


In [ ]:
import subprocess
import pandas as pd
import os
import sys
import json
import time
import re

# ==========================================
# CONFIGURATION
# ==========================================
SOLVER_TIME_LIMIT = 3600
ENABLE_BATCH_MODE = False
SINGLE_TARGET_INSTANCE = "98.txt"
n_50_signal = False

# --- LOGGING CONFIG ---
if n_50_signal:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n50"
    INPUT_CSV_PATH = "result_of_ea_TSP_dual_bounds_50_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSP_EA_dual_bound_verification_results_50_cus.csv"
    LOG_DIR = "solver_logs_TSP_EA_dual_bounds_50_cus"
else:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n20"
    INPUT_CSV_PATH = "result_of_ea_TSP_dual_bounds_20_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSP_EA_dual_bound_verification_results_20_cus.csv"
    LOG_DIR = "solver_logs_TSP_EA_dual_bounds_20_cus"

os.makedirs(LOG_DIR, exist_ok=True)

def run_verification_subprocess():
    print("🔹 Starting Subprocess Verification (with Node Count Recovery)...")
    
    if not os.path.exists(INPUT_CSV_PATH):
        print("❌ CSV missing.")
        return
    df_input = pd.read_csv(INPUT_CSV_PATH)
    
    targets = []
    if ENABLE_BATCH_MODE:
        processed = set()
        if os.path.exists(VERIFICATION_OUTPUT_CSV):
            try: processed = set(pd.read_csv(VERIFICATION_OUTPUT_CSV)['Instance'].values)
            except: pass
        for _, row in df_input.iterrows():
            if row['Instance'] not in processed:
                targets.append(row)
    else:
        row = df_input[df_input['Instance'] == SINGLE_TARGET_INSTANCE]
        if not row.empty: targets.append(row.iloc[0])

    print(f"📝 Queued {len(targets)} instances.")

    # --- MAIN LOOP ---
    for i, row in enumerate(targets):
        instance = row['Instance']
        chrom_str = row['Best_Chromosome']
        print(f"\n[{i+1}/{len(targets)}] Processing {instance}...")

        # 1. Setup Log File
        safe_name = instance.replace(".txt", "")
        log_file = os.path.join(LOG_DIR, f"solver_log_{safe_name}.txt")
        
        # 2. Construct Command
        cmd = [
            sys.executable, "-u", "worker.py", 
            instance, chrom_str, DATA_DIR, str(SOLVER_TIME_LIMIT)
        ]

        output_data = {}
        
        try:
            # 3. Run Process (Binary Mode)
            with open(log_file, "wb") as f:
                process = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
            
            # ==========================================
            # 4. PARSE LOGS (Crash or Success)
            # ==========================================
            # Initialize vars
            best_cost_found = None
            nodes_expanded = 0
            nodes_generated = "Unknown" # Cannot recover if crashed
            status_reason = "Unknown"

            # Scrape the text file
            try:
                with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        line = line.strip()
                        
                        # Recover Cost
                        match_bound = re.search(r"New primal bound:\s+(\d+(\.\d+)?)", line)
                        if match_bound:
                            best_cost_found = float(match_bound.group(1))
                        
                        # Recover Expanded Nodes
                        match_exp = re.search(r"expanded:\s+(\d+)", line)
                        if match_exp:
                            nodes_expanded = int(match_exp.group(1))
            except Exception as e:
                print(f"   ⚠️ Log Read Error: {e}")

            # ==========================================
            # 5. DETERMINE OUTCOME
            # ==========================================
            
            # CASE A: WORKER CRASHED (OOM or Error)
            if process.returncode != 0:
                print(f"⚠️ Worker Crashed (Code {process.returncode}).")
                
                if best_cost_found is not None:
                    print(f"   ✅ RECOVERED: Cost {best_cost_found}, Expanded {nodes_expanded}")
                    status_reason = "Crashed (Recovered)"
                    obj_val = f"{best_cost_found} (Crashed)"
                else:
                    print("   ❌ No result found in logs.")
                    status_reason = "Crashed (No Data)"
                    obj_val = "OOM/Error"

                output_data = {
                    "Instance": instance,
                    "Objective Value": obj_val,
                    "Optimality": False,
                    "Nodes Expanded": nodes_expanded,
                    "Nodes Generated": "Crash",
                    "Total Times (s)": "Crash",
                    "Bridging Time (s)": 0,
                    "Python Bound Time (s)": 0
                }

            # CASE B: WORKER FINISHED (Success or Timeout)
            else:
                # Try reading the final JSON line
                try:
                    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
                        lines = f.readlines()
                    
                    json_str = "{}"
                    for line in reversed(lines):
                        if line.strip().startswith("{"):
                            json_str = line.strip()
                            break
                    
                    res = json.loads(json_str)
                    
                    if res.get('status') == 'success':
                        print(f"   -> Success. Cost: {res['cost']}")
                        output_data = {
                            "Instance": instance,
                            "Objective Value": res['cost'],
                            "Optimality": res.get('is_optimal', False),
                            "Nodes Expanded": res.get('expanded', 0),
                            "Nodes Generated": res.get('generated', 0),
                            "Total Times (s)": res['duration'],
                            "Bridging Time (s)": res.get('stats', {}).get('total_bridge_time', 0),
                            "Python Bound Time (s)": res.get('stats', {}).get('python_dual_bound_calc_time', 0),
                        }
                    elif res.get('status') == 'timeout':
                        print(f"   ⚠️ Timeout (Managed).")
                        # Use recovered cost if JSON has inf
                        final_cost = res.get('cost')
                        if final_cost == float('inf') and best_cost_found is not None:
                            final_cost = f"{best_cost_found} (Timeout)"
                        
                        output_data = {
                            "Instance": instance, 
                            "Objective Value": final_cost,
                            "Optimality": False,
                            "Nodes Expanded": nodes_expanded, # Use recovered if JSON empty
                            "Total Times (s)": SOLVER_TIME_LIMIT
                        }
                    else:
                        print(f"   ⚠️ Worker Error Message: {res.get('message')}")
                        output_data = {"Instance": instance, "Objective Value": "Worker Error"}

                except json.JSONDecodeError:
                    print("   ⚠️ JSON Error. Using recovered text data.")
                    # Fallback to scraped text data
                    output_data = {
                        "Instance": instance,
                        "Objective Value": f"{best_cost_found} (JSON Error)" if best_cost_found else "Error",
                        "Nodes Expanded": nodes_expanded
                    }

            # ==========================================
            # 6. SAVE TO CSV
            # ==========================================
            df_new = pd.DataFrame([output_data])
            if os.path.exists(VERIFICATION_OUTPUT_CSV):
                df_existing = pd.read_csv(VERIFICATION_OUTPUT_CSV)
                # Remove old entry if exists to avoid duplicates
                if instance in df_existing['Instance'].values:
                    df_existing = df_existing[df_existing['Instance'] != instance]
                pd.concat([df_existing, df_new], ignore_index=True).to_csv(VERIFICATION_OUTPUT_CSV, index=False)
            else:
                df_new.to_csv(VERIFICATION_OUTPUT_CSV, index=False)
            print("   ✅ Saved to CSV.")

        except Exception as e:
            print(f"   ❌ Execution Failed: {e}")

    print("\n✅ Done.")

if __name__ == "__main__":
    run_verification_subprocess()

🔹 Starting Subprocess Verification (with Node Count Recovery)...
📝 Queued 1 instances.

[1/1] Processing 98.txt...
